<a id="top"></a>

# Demo: a tool call is just tokens

```yaml
title: "Demo: a tool call is just tokens"
keywords: tool call, tool_use block, tool definition, schema, reasoning is tokens, protocol, teacher demo
estimated duration: 10
```

> **Lesson:** L07. Teacher demo — Demo 1 in the [roadmap](../../../../docs/origin/lesson_roadmaps/L07/demos_or_activities.md). Makes **live** Claude calls — set `ANTHROPIC_API_KEY` (copy `.env.example` to `.env`) before running. A real model **varies**: dry-run before class and confirm the model reaches for the tool at least most of the time. Clear outputs before committing.
>
> **Why the raw Anthropic SDK here, not `potato_llm`?** The course's `potato_llm` seam is **text-only** — its `Message` carries a string and cannot represent `tool_use` / `tool_result` blocks. L07 is *about* those blocks, so these notebooks call the raw SDK directly. The API key still loads through the config seam (`require_anthropic_key`); we never hard-code it. This is the one lesson that legitimately reaches under the seam.
>
> The point to land: a `tool_use` block is **tokens the model emitted**, not an action it took. We stop the moment the block arrives and inspect it — we do **not** run the calculator.

## Contents

- [1. Setup](#1-setup)
- [2. No tools registered — the L01–L06 world](#2-no-tools-registered--the-l01l03-world)
- [3. Register the tool — and stop at the block](#3-register-the-tool--and-stop-at-the-block)
- [4. What just happened (and didn't)](#4-what-just-happened-and-didnt)
- [5. Takeaways](#5-takeaways)

## 1. Setup

The raw Anthropic client, the single `calculator` tool, and its tool definition. Anchor model: **Claude Sonnet 4.6**.

In [ ]:
import anthropic

from fluffy_potato_curriculum.common.config import require_anthropic_key

MODEL = "claude-sonnet-4-6"

# Constructing the client raises a clear error if ANTHROPIC_API_KEY is missing
# (the key is read through common.config, not a raw os.environ lookup).
client = anthropic.Anthropic(api_key=require_anthropic_key())


# The ONE tool that carries all four demos: a deterministic calculator.
# L07 is deliberately single-tool, single-round-trip (multi-tool is L08, the
# agent loop is L10). Resist adding a second tool.
def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression and return the result as a string.

    Restricted to digits and the operators + - * / ( ) . and whitespace so a
    hallucinated expression cannot run arbitrary code. Raises ValueError on
    anything else — that rejection is the whole point of Demo 4."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    # eval is safe here ONLY because the character whitelist above blocks names,
    # attributes, and calls. Never eval untrusted input without such a guard.
    return str(eval(expression))


# The tool DEFINITION: name + natural-language description + JSON-Schema input.
# This is all the model ever sees — never the Python function itself.
CALCULATOR_TOOL: dict[str, object] = {
    "name": "calculator",
    "description": (
        "Evaluate a basic arithmetic expression (the four operators and "
        "parentheses) and return the exact numeric result. Use this whenever "
        "the user asks for the result of a calculation."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "The arithmetic expression to evaluate, e.g. '18374 * 92431'.",
            }
        },
        "required": ["expression"],
    },
}


# A prompt the model is unreliable at without a calculator, so it has a real
# reason to reach for the tool.
PROMPT = "What is 18,374 multiplied by 92,431?"
print("setup ready (live model:", MODEL, ")")

[↑ Back to top](#top)

## 2. No tools registered — the L01–L06 world

First, the exact prompt with **no tools**. The model answers directly, often with a confident, wrong number. This is text-in, text-out — everything L01–L06 did.

In [ ]:
no_tools = client.messages.create(
    model=MODEL,
    max_tokens=200,
    messages=[{"role": "user", "content": PROMPT}],
)
for block in no_tools.content:
    print(block.type, "->", getattr(block, "text", block))

[↑ Back to top](#top)

## 3. Register the tool — and stop at the block

Now pass `tools=[CALCULATOR_TOOL]` and the **same** prompt. We stop as soon as the response arrives — we do **not** run the calculator. Print every content block so a `text` block and a `tool_use` block are visible side by side.

In [ ]:
resp = client.messages.create(
    model=MODEL,
    max_tokens=400,
    tools=[CALCULATOR_TOOL],
    messages=[{"role": "user", "content": PROMPT}],
)

print("stop_reason:", resp.stop_reason)
print("blocks in this one response:")
for block in resp.content:
    if block.type == "tool_use":
        print(f"  tool_use  name={block.name!r}  id={block.id!r}  input={block.input!r}")
    else:
        print(f"  {block.type}      {getattr(block, 'text', '')!r}")

[↑ Back to top](#top)

## 4. What just happened (and didn't)

Read the `tool_use` block aloud: a tool **name** (`calculator`), an **arguments** object (`{'expression': ...}`), and a unique **id**. Nothing was computed — the model did not multiply anything. It emitted a *block of tokens that says* 'I would like the calculator run with this expression.' The number inside the arguments came from the model's tokens, not from arithmetic.

In [ ]:
tool_use = next(b for b in resp.content if b.type == "tool_use")
print("the model PROPOSED this call (it did not run it):")
print("  name      :", tool_use.name)
print("  arguments :", tool_use.input)
print("  call id   :", tool_use.id)
print("\nThree actors, only one has acted: the MODEL proposed; the APPLICATION (us)")
print("holds the block and has done nothing; the TOOL (calculator) has not run.")

[↑ Back to top](#top)

## 5. Takeaways

- A `tool_use` block is **just more structured tokens** — the same shape-contract idea as L06's `<thinking>`/`<answer>` tags, except now the application is expected to *act* on the shape.
- **Three actors**, only one has moved: model proposed, application holds the block, tool has not run. The rest of the lesson fills in the remaining steps.
- The decision to emit a tool call *instead of* answering directly is itself a **reasoning step** (reinforce L06 — don't re-teach). A vague tool is one the model skips ([L08](L0402_lecture.md) answers *why*).
- Next: [Demo 2](L0404_lecture.ipynb) completes the loop this demo stopped halfway through.

[↑ Back to top](#top)